In [1]:
# ============================================================
# lv 1 - 환경변수 / 라이브러리
# ============================================================

import os
from dotenv import load_dotenv

import fitz

from collections import Counter

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient, models
from qdrant_client.http.models import (
    Distance,
    VectorParams,
    PayloadSchemaType,
)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


load_dotenv()


# ------------------------------------------------------------
# API Key 확인
# ------------------------------------------------------------

if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")


# ------------------------------------------------------------
# Qdrant Cloud 설정 확인
# ------------------------------------------------------------

if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")


# ============================================================
# lv 2 - 페이지별 카테고리 분류
# ============================================================

def get_category(page_num):
    """
    2026_freshman_c.pdf의 PDF 물리 페이지 번호 기준 분류.
    """
    if 1 <= page_num <= 9:
        return "기타"
    elif 10 <= page_num <= 22:
        if 10 <= page_num <= 16:
            return "졸업_및_수강신청_안내"
        elif 17 <= page_num <= 18:
            return "출결제도_안내"
        elif page_num == 19:
            return "학사제도_안내"
        elif 20 <= page_num <= 21:
            return "학적변동_및_각종_증명서_발급_안내"
        elif page_num == 22:
            return "학부_과_전공_사무실_전화번호_안내"
        else:
            return "교무팀"
    elif 23 <= page_num <= 24:
        if page_num == 23:
            return "복합__SM_IN_핵심역량__SM_IN_핵심역량_진단_및_인증"
        elif page_num == 24:
            return "학생_참여_프로그램"
        else:
            return "교육혁신추진팀"
    elif 25 <= page_num <= 27:
        if page_num == 25:
            return "복합__전공탐색_교과목__교원_학생끌어주기_프로그램__선후배_이어주기_프로그램__전공선택_징검다리"
        elif page_num == 26:
            return "자유전공생_전용_공간_코워킹스페이스_Coworking_Space"
        elif page_num == 27:
            return "자유전공학부생_교육과정"
        else:
            return "자유전공학부지원센터"
    elif 28 <= page_num <= 29:
        return "비교과교육과정"
    elif 30 <= page_num <= 33:
        return "지능형로봇_혁신융합대학"
    elif 34 <= page_num <= 37:
        if 34 <= page_num <= 35:
            return "2026학년도_신입생_적용_교양_교육과정_이수_원칙"
        elif 36 <= page_num <= 37:
            return "계당교양교육원_의사소통능력개발센터_비교과_프로그램_안내"
        else:
            return "계당교양교육원"
    elif 38 <= page_num <= 45:
        if page_num == 38:
            return "복합__학생증_발급__복지시설_현황"
        elif page_num == 39:
            return "복합__통학버스_및_무료_셔틀버스_운행_안내__학생_대상_안전_보험_안내"
        elif 40 <= page_num <= 45:
            return "장학금_학자금대출_제도_안내"
        else:
            return "학생복지팀"
    elif page_num == 46:
        return "장애학생_지원_제도"
    elif 47 <= page_num <= 48:
        return "기타"
    elif 49 <= page_num <= 50:
        return "취업능력_향상_교육_훈련_및_진로_설정_프로그램_운영"
    elif 51 <= page_num <= 52:
        return "현장_교육_프로그램"
    elif 53 <= page_num <= 61:
        if 53 <= page_num <= 55:
            return "시설안내"
        elif page_num == 56:
            return "자료대출_및_반납"
        elif 57 <= page_num <= 59:
            return "이용자_서비스"
        elif page_num == 60:
            return "복합__e_Book_전자잡지__e_Learning"
        elif page_num == 61:
            return "학술정보관_모바일_어플리케이션"
        else:
            return "학술정보관"
    elif 62 <= page_num <= 63:
        return "국제교류프로그램_안내"
    elif 64 <= page_num <= 65:
        return "주요업무"
    elif 66 <= page_num <= 73:
        if page_num == 66:
            return "공용_컴퓨터_실습실_현황"
        elif 67 <= page_num <= 69:
            return "샘물_포탈서비스_접속_및_학생메일_Office365_설치"
        elif page_num == 70:
            return "무선랜_인증_방법_및_이용_안내"
        elif page_num == 71:
            return "안드로이드_SANGMYUNG_무선랜_네트워크_설정_방법"
        elif page_num == 72:
            return "iPhone_SANGMYUNG_무선랜_네트워크_설정_방법"
        elif page_num == 73:
            return "불법_소프트웨어_사용_금지_안내"
        else:
            return "정보통신지원팀"
    elif 74 <= page_num <= 78:
        if page_num == 74:
            return "일반대학원_서울"
        elif page_num == 75:
            return "일반대학원_천안"
        elif page_num == 76:
            return "특수대학원_서울"
        elif 77 <= page_num <= 78:
            return "학_석사연계과정_지원안내"
        else:
            return "대학원교학팀"
    elif 79 <= page_num <= 81:
        if page_num == 79:
            return "학생상담센터_주요_프로그램"
        elif 80 <= page_num <= 81:
            return "상담_신청절차_및_이용_방법"
        else:
            return "학생상담센터"
    elif 82 <= page_num <= 83:
        if page_num == 82:
            return "상담_및_사건처리_절차"
        elif page_num == 83:
            return "복합__온라인_폭력예방통합교육_이수_안내__인권센터_이용안내"
        else:
            return "인권센터"
    elif 84 <= page_num <= 85:
        return "학생생활관_현황"
    elif 86 <= page_num <= 96:
        if page_num == 86:
            return "신입생_모집안내"
        elif 87 <= page_num <= 89:
            return "사용안내"
        elif 90 <= page_num <= 91:
            return "본관_시설"
        elif page_num == 92:
            return "복합__체육관_및_다목적강당_시설__야외_시설"
        elif page_num == 93:
            return "복합__여가_시설__부엉이박물관"
        elif 94 <= page_num <= 96:
            return "상운관"
        else:
            return "상명수련원"
    elif 97 <= page_num <= 105:
        if page_num == 97:
            return "병역판정검사_입영_및_연기"
        elif 98 <= page_num <= 100:
            return "대학직장예비군_편성_및_교육훈련"
        elif 101 <= page_num <= 103:
            return "민방위_경계_공습_경보시_행동요령"
        elif page_num == 104:
            return "화재발생시_행동요령"
        elif page_num == 105:
            return "복합__공연관람시_행동요령__낙뢰시_행동요령__태풍시_행동요령"
        else:
            return "예비군대대"
    elif 106 <= page_num <= 108:
        return "학군사관_후보생_선발_교육"
    elif 109 <= page_num <= 110:
        if page_num == 109:
            return "복합__시설현황__운영_안내"
        elif page_num == 110:
            return "복합__기타_운영사항__특이사항"
        else:
            return "상명스포츠센터"
    elif page_num == 111:
        return "복합__건물_출입문_개폐시간_안내__상명대학교_전동_킥보드_및_자전거_PM_운행_및_주차_구역_안내도"
    elif page_num == 112:
        return "우편취급국_이용_안내"
    elif page_num == 113:
        return "보건건강관리센터_이용_안내"
    elif 114 <= page_num <= 115:
        return "기타"
    elif page_num == 116:
        return "학군사관_후보생_선발_교육"
    else:
        return "기타"


# ============================================================
# lv 3 - PDF 읽기 → LangChain Document 생성
# ============================================================

file_path = "../datasets/2026_freshman_c.pdf"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {file_path}")

pdf = fitz.open(file_path)
docs = []

for page_index in range(len(pdf)):
    page = pdf[page_index]
    page_num = page_index + 1
    text = page.get_text("text", sort=True).strip()

    if not text:
        continue

    category = get_category(page_num)

    if category.startswith("복합__"):
        category_list = category.replace("복합__", "", 1).split("__")
    else:
        category_list = [category]

    document = Document(
        page_content=text,
        metadata={
            "source": os.path.basename(file_path),
            "page": page_num,
            "category": category,
            "categories": category_list,
            "year": 2026,
        }
    )
    docs.append(document)

pdf.close()
print(f"✓ 총 {len(docs)}개 문서 생성 완료")


# ============================================================
# lv 4 - 카테고리 검수 (요약 출력)
# ============================================================

categories = [doc.metadata["category"] for doc in docs]
category_counts = Counter(categories)

print(f"✓ 전체 Document 수: {len(docs)}개 / 카테고리 종류: {len(category_counts)}개")

# 116개 전체 반복 출력을 축소하여 생략 문제 방지 (상위 5개 및 주요 카테고리만 표기)
print("  [카테고리 요약 현황]")
for category, count in list(sorted(category_counts.items()))[:5]:
    print(f"   - {category}: {count}개")
print("   ... (이하 생략)")


# ============================================================
# lv 5 - Qdrant Cloud 연결
# ============================================================

client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)
print("✓ Qdrant Cloud 연결 완료")


# ============================================================
# lv 6 - Embedding / Collection 생성
# ============================================================

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
collection_name = "freshman_2026_metadata"

collections = client.get_collections().collections
existing_collection = any(collection.name == collection_name for collection in collections)

should_ingest = True

if existing_collection:
    print(f"\n컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == "y":
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("✓ 기존 컬렉션 삭제 완료")
    else:
        print("기존 컬렉션을 그대로 사용합니다.")
        should_ingest = False

if not existing_collection or should_ingest:
    if should_ingest:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"✓ 컬렉션 '{collection_name}' 생성 완료")

        client.create_payload_index(collection_name=collection_name, field_name="metadata.category", field_schema=PayloadSchemaType.KEYWORD)
        client.create_payload_index(collection_name=collection_name, field_name="metadata.categories", field_schema=PayloadSchemaType.KEYWORD)
        client.create_payload_index(collection_name=collection_name, field_name="metadata.page", field_schema=PayloadSchemaType.INTEGER)
        client.create_payload_index(collection_name=collection_name, field_name="metadata.source", field_schema=PayloadSchemaType.KEYWORD)
        client.create_payload_index(collection_name=collection_name, field_name="metadata.year", field_schema=PayloadSchemaType.INTEGER)
        print("✓ 메타데이터 인덱스 생성 완료")


# ============================================================
# lv 7 - Vector Store 생성 + 데이터 저장
# ============================================================

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

if should_ingest:
    print(f"\n{len(docs)}개 문서 임베딩 및 Qdrant 저장 시작...")
    vectorstore.add_documents(documents=docs)
    print(f"✓ {len(docs)}개 문정이 Qdrant Cloud에 추가되었습니다.")


# ============================================================
# lv 8 - RAG 체인 구축 및 최종 답변 생성
# ============================================================

search_category = "학생증_발급"
search_query = "학생증은 어떻게 발급받나요?"

filter_condition = models.Filter(
    must=[
        models.FieldCondition(
            key="metadata.categories",
            match=models.MatchValue(value=search_category)
        )
    ]
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": filter_condition
    }
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt_template = """다음 제공된 문맥(Context)만을 바탕으로 질문에 정확하고 친절하게 답변해주세요.
문맥에서 답을 찾을 수 없다면 "제공된 문서에서 관련 정보를 찾을 수 없습니다."라고 답하세요.

[문맥 정보]
{context}

[질문]
{question}

[답변]"""

prompt = ChatPromptTemplate.from_template(prompt_template)

def format_docs(docs):
    if not docs:
        return "검색된 문서가 없습니다."
    return "\n\n".join(
        f"[PDF {doc.metadata.get('page')}페이지]\n{doc.page_content}" 
        for doc in docs
    )

rag_chain = (
    {
        "context": retriever | format_docs, 
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# ------------------------------------------------------------
# 최종 결과 출력
# ------------------------------------------------------------
print("\n" + "=" * 80)
print(f"❓ 질문 (Query)      : {search_query}")
print(f"🏷️  카테고리 필터    : {search_category}")
print("=" * 80)

retrieved_docs = retriever.invoke(search_query)
print(f"\n📄 참고한 관련 문서 수: {len(retrieved_docs)}개")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"   {i}. PDF {doc.metadata.get('page')}페이지 (카테고리: {doc.metadata.get('category')})")

print("\n" + "-" * 50)
print("🤖 [AI 최종 답변]")
print("-" * 50)

response = rag_chain.invoke(search_query)
print(response)

print("\n" + "=" * 80 + "\n")
print("✓ 전체 작업 완료")

✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://f9ea8747-5ece-4c5d-9ce8-a9b7f6381aa9.us-east-2-0.aws.cloud.qdrant.io:6333
✓ 총 116개 문서 생성 완료
✓ 전체 Document 수: 116개 / 카테고리 종류: 60개
  [카테고리 요약 현황]
   - 2026학년도_신입생_적용_교양_교육과정_이수_원칙: 2개
   - iPhone_SANGMYUNG_무선랜_네트워크_설정_방법: 1개
   - 계당교양교육원_의사소통능력개발센터_비교과_프로그램_안내: 2개
   - 공용_컴퓨터_실습실_현황: 1개
   - 국제교류프로그램_안내: 2개
   ... (이하 생략)
✓ Qdrant Cloud 연결 완료

컬렉션 'freshman_2026_metadata'이 이미 존재합니다.
컬렉션 'freshman_2026_metadata' 삭제 중...
✓ 기존 컬렉션 삭제 완료
✓ 컬렉션 'freshman_2026_metadata' 생성 완료
✓ 메타데이터 인덱스 생성 완료

116개 문서 임베딩 및 Qdrant 저장 시작...
✓ 116개 문정이 Qdrant Cloud에 추가되었습니다.

❓ 질문 (Query)      : 학생증은 어떻게 발급받나요?
🏷️  카테고리 필터    : 학생증_발급

📄 참고한 관련 문서 수: 1개
   1. PDF 38페이지 (카테고리: 복합__학생증_발급__복지시설_현황)

--------------------------------------------------
🤖 [AI 최종 답변]
--------------------------------------------------
학생증은 다음과 같은 방법으로 발급받을 수 있습니다.

1. **대상**: 2026학년도 신편입생
2. **신청기간**: 2026년 3월 3일부터 상시 가능
3. **신청방법**: 대학 홈페이지에서 '대학생활' → '학생지원' → '학생증발

In [2]:
# ============================================================
# lv 1 - 환경변수 / 라이브러리
# ============================================================

import os
from dotenv import load_dotenv

import fitz

from collections import Counter

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient, models
from qdrant_client.http.models import (
    Distance,
    VectorParams,
    PayloadSchemaType,
)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


load_dotenv()


# ------------------------------------------------------------
# API Key 확인
# ------------------------------------------------------------

if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")


# ------------------------------------------------------------
# Qdrant Cloud 설정 확인
# ------------------------------------------------------------

if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")


# ============================================================
# lv 2 - 페이지별 카테고리 분류
# ============================================================

def get_category(page_num):
    """
    2026_freshman_c.pdf의 PDF 물리 페이지 번호 기준 분류.
    """
    if 1 <= page_num <= 9:
        return "기타"
    elif 10 <= page_num <= 22:
        if 10 <= page_num <= 16:
            return "졸업_및_수강신청_안내"
        elif 17 <= page_num <= 18:
            return "출결제도_안내"
        elif page_num == 19:
            return "학사제도_안내"
        elif 20 <= page_num <= 21:
            return "학적변동_및_각종_증명서_발급_안내"
        elif page_num == 22:
            return "학부_과_전공_사무실_전화번호_안내"
        else:
            return "교무팀"
    elif 23 <= page_num <= 24:
        if page_num == 23:
            return "복합__SM_IN_핵심역량__SM_IN_핵심역량_진단_및_인증"
        elif page_num == 24:
            return "학생_참여_프로그램"
        else:
            return "교육혁신추진팀"
    elif 25 <= page_num <= 27:
        if page_num == 25:
            return "복합__전공탐색_교과목__교원_학생끌어주기_프로그램__선후배_이어주기_프로그램__전공선택_징검다리"
        elif page_num == 26:
            return "자유전공생_전용_공간_코워킹스페이스_Coworking_Space"
        elif page_num == 27:
            return "자유전공학부생_교육과정"
        else:
            return "자유전공학부지원센터"
    elif 28 <= page_num <= 29:
        return "비교과교육과정"
    elif 30 <= page_num <= 33:
        return "지능형로봇_혁신융합대학"
    elif 34 <= page_num <= 37:
        if 34 <= page_num <= 35:
            return "2026학년도_신입생_적용_교양_교육과정_이수_원칙"
        elif 36 <= page_num <= 37:
            return "계당교양교육원_의사소통능력개발센터_비교과_프로그램_안내"
        else:
            return "계당교양교육원"
    elif 38 <= page_num <= 45:
        if page_num == 38:
            return "복합__학생증_발급__복지시설_현황"
        elif page_num == 39:
            return "복합__통학버스_및_무료_셔틀버스_운행_안내__학생_대상_안전_보험_안내"
        elif 40 <= page_num <= 45:
            return "장학금_학자금대출_제도_안내"
        else:
            return "학생복지팀"
    elif page_num == 46:
        return "장애학생_지원_제도"
    elif 47 <= page_num <= 48:
        return "기타"
    elif 49 <= page_num <= 50:
        return "취업능력_향상_교육_훈련_및_진로_설정_프로그램_운영"
    elif 51 <= page_num <= 52:
        return "현장_교육_프로그램"
    elif 53 <= page_num <= 61:
        if 53 <= page_num <= 55:
            return "시설안내"
        elif page_num == 56:
            return "자료대출_및_반납"
        elif 57 <= page_num <= 59:
            return "이용자_서비스"
        elif page_num == 60:
            return "복합__e_Book_전자잡지__e_Learning"
        elif page_num == 61:
            return "학술정보관_모바일_어플리케이션"
        else:
            return "학술정보관"
    elif 62 <= page_num <= 63:
        return "국제교류프로그램_안내"
    elif 64 <= page_num <= 65:
        return "주요업무"
    elif 66 <= page_num <= 73:
        if page_num == 66:
            return "공용_컴퓨터_실습실_현황"
        elif 67 <= page_num <= 69:
            return "샘물_포탈서비스_접속_및_학생메일_Office365_설치"
        elif page_num == 70:
            return "무선랜_인증_방법_및_이용_안내"
        elif page_num == 71:
            return "안드로이드_SANGMYUNG_무선랜_네트워크_설정_방법"
        elif page_num == 72:
            return "iPhone_SANGMYUNG_무선랜_네트워크_설정_방법"
        elif page_num == 73:
            return "불법_소프트웨어_사용_금지_안내"
        else:
            return "정보통신지원팀"
    elif 74 <= page_num <= 78:
        if page_num == 74:
            return "일반대학원_서울"
        elif page_num == 75:
            return "일반대학원_천안"
        elif page_num == 76:
            return "특수대학원_서울"
        elif 77 <= page_num <= 78:
            return "학_석사연계과정_지원안내"
        else:
            return "대학원교학팀"
    elif 79 <= page_num <= 81:
        if page_num == 79:
            return "학생상담센터_주요_프로그램"
        elif 80 <= page_num <= 81:
            return "상담_신청절차_및_이용_방법"
        else:
            return "학생상담센터"
    elif 82 <= page_num <= 83:
        if page_num == 82:
            return "상담_및_사건처리_절차"
        elif page_num == 83:
            return "복합__온라인_폭력예방통합교육_이수_안내__인권센터_이용안내"
        else:
            return "인권센터"
    elif 84 <= page_num <= 85:
        return "학생생활관_현황"
    elif 86 <= page_num <= 96:
        if page_num == 86:
            return "신입생_모집안내"
        elif 87 <= page_num <= 89:
            return "사용안내"
        elif 90 <= page_num <= 91:
            return "본관_시설"
        elif page_num == 92:
            return "복합__체육관_및_다목적강당_시설__야외_시설"
        elif page_num == 93:
            return "복합__여가_시설__부엉이박물관"
        elif 94 <= page_num <= 96:
            return "상운관"
        else:
            return "상명수련원"
    elif 97 <= page_num <= 105:
        if page_num == 97:
            return "병역판정검사_입영_및_연기"
        elif 98 <= page_num <= 100:
            return "대학직장예비군_편성_및_교육훈련"
        elif 101 <= page_num <= 103:
            return "민방위_경계_공습_경보시_행동요령"
        elif page_num == 104:
            return "화재발생시_행동요령"
        elif page_num == 105:
            return "복합__공연관람시_행동요령__낙뢰시_행동요령__태풍시_행동요령"
        else:
            return "예비군대대"
    elif 106 <= page_num <= 108:
        return "학군사관_후보생_선발_교육"
    elif 109 <= page_num <= 110:
        if page_num == 109:
            return "복합__시설현황__운영_안내"
        elif page_num == 110:
            return "복합__기타_운영사항__특이사항"
        else:
            return "상명스포츠센터"
    elif page_num == 111:
        return "복합__건물_출입문_개폐시간_안내__상명대학교_전동_킥보드_및_자전거_PM_운행_및_주차_구역_안내도"
    elif page_num == 112:
        return "우편취급국_이용_안내"
    elif page_num == 113:
        return "보건건강관리센터_이용_안내"
    elif 114 <= page_num <= 115:
        return "기타"
    elif page_num == 116:
        return "학군사관_후보생_선발_교육"
    else:
        return "기타"


# ============================================================
# lv 3 - PDF 읽기 → LangChain Document 생성
# ============================================================

file_path = "../datasets/2026_freshman_c.pdf"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {file_path}")

pdf = fitz.open(file_path)
docs = []

for page_index in range(len(pdf)):
    page = pdf[page_index]
    page_num = page_index + 1
    text = page.get_text("text", sort=True).strip()

    if not text:
        continue

    category = get_category(page_num)

    if category.startswith("복합__"):
        category_list = category.replace("복합__", "", 1).split("__")
    else:
        category_list = [category]

    document = Document(
        page_content=text,
        metadata={
            "source": os.path.basename(file_path),
            "page": page_num,
            "category": category,
            "categories": category_list,
            "year": 2026,
        }
    )
    docs.append(document)

pdf.close()
print(f"✓ 총 {len(docs)}개 문서 생성 완료")


# ============================================================
# lv 4 - 카테고리 검수 (요약 출력)
# ============================================================

categories = [doc.metadata["category"] for doc in docs]
category_counts = Counter(categories)

print(f"✓ 전체 Document 수: {len(docs)}개 / 카테고리 종류: {len(category_counts)}개")

# 116개 전체 반복 출력을 축소하여 생략 문제 방지 (상위 5개 및 주요 카테고리만 표기)
print("  [카테고리 요약 현황]")
for category, count in list(sorted(category_counts.items()))[:5]:
    print(f"   - {category}: {count}개")
print("   ... (이하 생략)")


# ============================================================
# lv 5 - Qdrant Cloud 연결
# ============================================================

client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)
print("✓ Qdrant Cloud 연결 완료")


# ============================================================
# lv 6 - Embedding / Collection 생성
# ============================================================

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
collection_name = "freshman_2026_metadata"

collections = client.get_collections().collections
existing_collection = any(collection.name == collection_name for collection in collections)

should_ingest = True

if existing_collection:
    print(f"\n컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == "y":
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("✓ 기존 컬렉션 삭제 완료")
    else:
        print("기존 컬렉션을 그대로 사용합니다.")
        should_ingest = False

if not existing_collection or should_ingest:
    if should_ingest:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"✓ 컬렉션 '{collection_name}' 생성 완료")

        client.create_payload_index(collection_name=collection_name, field_name="metadata.category", field_schema=PayloadSchemaType.KEYWORD)
        client.create_payload_index(collection_name=collection_name, field_name="metadata.categories", field_schema=PayloadSchemaType.KEYWORD)
        client.create_payload_index(collection_name=collection_name, field_name="metadata.page", field_schema=PayloadSchemaType.INTEGER)
        client.create_payload_index(collection_name=collection_name, field_name="metadata.source", field_schema=PayloadSchemaType.KEYWORD)
        client.create_payload_index(collection_name=collection_name, field_name="metadata.year", field_schema=PayloadSchemaType.INTEGER)
        print("✓ 메타데이터 인덱스 생성 완료")


# ============================================================
# lv 7 - Vector Store 생성 + 데이터 저장
# ============================================================

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

if should_ingest:
    print(f"\n{len(docs)}개 문서 임베딩 및 Qdrant 저장 시작...")
    vectorstore.add_documents(documents=docs)
    print(f"✓ {len(docs)}개 문정이 Qdrant Cloud에 추가되었습니다.")


# ============================================================
# lv 8 - RAG 체인 구축 및 최종 답변 생성
# ============================================================

search_category = "학생증_발급"
search_query = "학생증은 어떻게 발급받나요?"

filter_condition = models.Filter(
    must=[
        models.FieldCondition(
            key="metadata.categories",
            match=models.MatchValue(value=search_category)
        )
    ]
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": filter_condition
    }
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt_template = """다음 제공된 문맥(Context)만을 바탕으로 질문에 정확하고 친절하게 답변해주세요.
문맥에서 답을 찾을 수 없다면 "제공된 문서에서 관련 정보를 찾을 수 없습니다."라고 답하세요.

[문맥 정보]
{context}

[질문]
{question}

[답변]"""

prompt = ChatPromptTemplate.from_template(prompt_template)

def format_docs(docs):
    if not docs:
        return "검색된 문서가 없습니다."
    return "\n\n".join(
        f"[PDF {doc.metadata.get('page')}페이지]\n{doc.page_content}" 
        for doc in docs
    )

rag_chain = (
    {
        "context": retriever | format_docs, 
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# ------------------------------------------------------------
# 최종 결과 출력
# ------------------------------------------------------------
print("\n" + "=" * 80)
print(f"❓ 질문 (Query)      : {search_query}")
print(f"🏷️  카테고리 필터    : {search_category}")
print("=" * 80)

retrieved_docs = retriever.invoke(search_query)
print(f"\n📄 참고한 관련 문서 수: {len(retrieved_docs)}개")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"   {i}. PDF {doc.metadata.get('page')}페이지 (카테고리: {doc.metadata.get('category')})")

print("\n" + "-" * 50)
print("🤖 [AI 최종 답변]")
print("-" * 50)

response = rag_chain.invoke(search_query)
print(response)

print("\n" + "=" * 80 + "\n")
print("✓ 전체 작업 완료")


✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://f9ea8747-5ece-4c5d-9ce8-a9b7f6381aa9.us-east-2-0.aws.cloud.qdrant.io:6333
✓ 총 116개 문서 생성 완료
✓ 전체 Document 수: 116개 / 카테고리 종류: 60개
  [카테고리 요약 현황]
   - 2026학년도_신입생_적용_교양_교육과정_이수_원칙: 2개
   - iPhone_SANGMYUNG_무선랜_네트워크_설정_방법: 1개
   - 계당교양교육원_의사소통능력개발센터_비교과_프로그램_안내: 2개
   - 공용_컴퓨터_실습실_현황: 1개
   - 국제교류프로그램_안내: 2개
   ... (이하 생략)
✓ Qdrant Cloud 연결 완료

컬렉션 'freshman_2026_metadata'이 이미 존재합니다.
컬렉션 'freshman_2026_metadata' 삭제 중...
✓ 기존 컬렉션 삭제 완료
✓ 컬렉션 'freshman_2026_metadata' 생성 완료
✓ 메타데이터 인덱스 생성 완료

116개 문서 임베딩 및 Qdrant 저장 시작...
✓ 116개 문정이 Qdrant Cloud에 추가되었습니다.

❓ 질문 (Query)      : 학생증은 어떻게 발급받나요?
🏷️  카테고리 필터    : 학생증_발급

📄 참고한 관련 문서 수: 1개
   1. PDF 38페이지 (카테고리: 복합__학생증_발급__복지시설_현황)

--------------------------------------------------
🤖 [AI 최종 답변]
--------------------------------------------------
학생증은 다음과 같은 방법으로 발급받을 수 있습니다.

1. **대상**: 2026학년도 신편입생
2. **신청기간**: 2026년 3월 3일부터 상시 가능
3. **신청방법**: 대학 홈페이지에서 '대학생활' → '학생지원' → '학생증발

In [32]:
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Supabase 설정 확인
if os.environ.get("SUPABASE_DB_URL"):
    print("✓ Supabase DB URL이 설정되었습니다.")
else:
    print("✗ Supabase DB URL이 필요합니다.")
    print("  .env 파일에 SUPABASE_DB_URL을 추가하세요.")

import pandas as pd

# TODO: 팀의 CSV 파일 경로를 입력하세요
# 여러 파일이 있다면 dict 형태로 구성
# 예시:
# csv_files = {
#     "table1": "../datasets/your_table1.csv",
#     "table2": "../datasets/your_table2.csv"
# }

csv_files = {
    "충청남도_천안시_착한가격업소": "충청남도_천안시_착한가격업소_20260326.csv"
    # 필요한 만큼 추가
}

# CSV 파일 로드 및 확인
dataframes = {}

for table_name, file_path in csv_files.items():
    try:
        df = pd.read_csv(file_path, encoding='cp949')
        dataframes[table_name] = df

        print("=" * 80)
        print(f"📋 {table_name} 테이블")
        print("=" * 80)
        print(f"\n행 수: {len(df)}")
        print(f"컬럼: {list(df.columns)}")
        print(f"\n첫 5개 행:")
        print(df.head())
        print(f"\n데이터 타입:")
        print(df.dtypes)
        print("\n")

    except Exception as e:
        print(f"✗ {table_name} 로드 실패: {e}\n")

print(f"\n✓ 총 {len(dataframes)}개의 테이블 로드 완료")

from langchain_community.utilities import SQLDatabase

supabase_db_url = os.getenv("SUPABASE_DB_URL")

if not supabase_db_url:
    raise Exception("SUPABASE_DB_URL이 설정되지 않았습니다. .env 파일을 확인하세요.")

print("Supabase PostgreSQL 연결 중...\n")

try:
    # LangChain SQLDatabase로 PostgreSQL 연결
    db = SQLDatabase.from_uri(
    supabase_db_url,
    include_tables=["충청남도 천안시_착한가격업소_20260326"]
)

    print("✓ PostgreSQL 연결 성공!\n")
    print("현재 테이블 목록:")
    tables = db.get_usable_table_names()
    print(tables)

except Exception as e:
    print(f"✗ 연결 실패: {e}")
    raise

# 데이터베이스 다시 연결 (업로드 후 스키마 갱신)
db = SQLDatabase.from_uri(
    supabase_db_url,
    include_tables=["충청남도 천안시_착한가격업소_20260326"]
)

print("=== 데이터베이스 스키마 ===")
print(db.table_info)

print("\n" + "="*80 + "\n")

# 각 테이블의 샘플 데이터
for table in db.get_usable_table_names():
    print(f"[{table}] 테이블 샘플:")
    try:
        result = db.run(f'SELECT * FROM "{table}" LIMIT 3;')
        print(result)
    except Exception as e:
        print(f"조회 실패: {e}")
    print()

# TODO: 기본 조회 쿼리 작성
# 예시: 특정 조건으로 데이터 조회

query = 'SELECT "업소명", "업종", "소재지주소" FROM "충청남도 천안시_착한가격업소_20260326" WHERE "업종" = \'한식\' LIMIT 5;'

print("실행 쿼리:")
print(query)
print("\n결과:")

try:
    result = db.run(query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

# TODO: JOIN 쿼리 작성 (여러 테이블을 사용하는 경우)
# 예시: 두 테이블을 조인하여 데이터 조회

join_query = 'SELECT "업종", COUNT(*) FROM "충청남도 천안시_착한가격업소_20260326" GROUP BY "업종";'

print("실행 쿼리:")
print(join_query)
print("\n결과:")

try:
    result = db.run(join_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

llm = init_chat_model("gpt-5.4-mini")

def text_to_sql(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문을 SQL로 변환
    """
    # TODO: 시스템 프롬프트를 팀 데이터에 맞게 수정하세요
    system_prompt = f"""
    당신은 SQL 전문가입니다.
    사용자의 질문을 SQL 쿼리로 변환하세요.

    데이터베이스 스키마:
    {db.table_info}

    규칙:
    - PostgreSQL 문법 사용
    - SELECT 쿼리만 생성 (INSERT, UPDATE, DELETE 금지)
    - SQL 코드만 반환 (설명 불필요)
    - 코드 블록(```) 없이 순수 SQL만 반환
    - 세미콜론(;)으로 끝내기

    사용 가능한 SQL 문법:
    - JOIN (INNER, LEFT, RIGHT, FULL)
    - GROUP BY, HAVING
    - 집계 함수 (COUNT, SUM, AVG, MIN, MAX)
    - 서브쿼리
    - WHERE, ORDER BY, LIMIT
    - CTE (WITH 절)
    - 윈도우 함수
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ]

    response = llm.invoke(messages)
    sql = response.content.strip()

    # 코드 블록 제거
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
        sql = sql.replace("sql", "").replace("```", "").strip()

    return sql

print("✓ Text2SQL 함수 준비 완료")



def query_database(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문 → SQL 생성 → 실행 → 자연어 답변
    """
    # 1. SQL 생성
    print(f"[1] SQL 생성 중...")
    sql = text_to_sql(question, db)
    print(f"    {sql}\n")

    # 2. SQL 실행
    print(f"[2] SQL 실행 중...")
    try:
        result = db.run(sql)
        print(f"    실행 완료\n")
    except Exception as e:
        return f"SQL 실행 오류: {e}"

    # 3. 자연어 답변 생성
    print(f"[3] 답변 생성 중...")

    # TODO: 시스템 프롬프트를 팀 데이터 도메인에 맞게 수정하세요
    system_prompt = """
    당신은 [YOUR_DOMAIN] 데이터 분석 전문가입니다.
    SQL 쿼리 결과를 기반으로 사용자의 질문에 자연스럽게 답변하세요.
    답변은 명확하고 이해하기 쉽게 작성하세요.
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
        질문: {question}

        실행한 SQL:
        {sql}

        쿼리 결과:
        {result}

        위 결과를 바탕으로 질문에 답변해주세요.
        """)
    ]

    response = llm.invoke(messages)
    return response.content

print("✓ 완전한 Text2SQL 시스템 준비 완료")

from IPython.display import Markdown, display

# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "YOUR_QUESTION_HERE"

print(f"질문: {question}\n")
print("="*80 + "\n")

answer = query_database(question, db)

print("\n" + "="*80)
print("\n답변:")
display(Markdown(answer))

# TODO: 팀 데이터에 맞는 다양한 질문들을 작성하세요
# 기본 조회, JOIN, 집계, 정렬 등 다양한 유형의 질문 포함

questions = [
    "천안시 착한가격 업소는 총 몇개인가요?",
    "천안시 착한가격 업소 중 한식 업종은 몇개인가요?",
    "천안시 착한가격 업소 중 중식 업종의 상호명과 주소를 알려주세요.",

]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print("="*80 + "\n")

    try:
        answer = query_database(q, db)
        print("\n답변:")
        display(Markdown(answer))
    except Exception as e:
        print(f"오류: {e}")
        

✓ OpenAI API Key가 설정되었습니다.
✓ Supabase DB URL이 설정되었습니다.
📋 충청남도_천안시_착한가격업소 테이블

행 수: 124
컬럼: ['연번', '지정번호', '업소명', '업종', '대표자', '읍면동', '행정동', '소재지주소', '데이터기준일자']

첫 5개 행:
   연번         지정번호        업소명    업종  대표자  읍면동   행정동  \
0   1  천안-2011-005     선비숯불갈비    한식  홍대의  신부동   신안동   
1   2  천안-2012-020     포인트미용실  이미용업  원용백  대흥동   중앙동   
2   3  천안-2012-021      스타미용실  이미용업  신필자  대흥동   중앙동   
3   4  천안-2012-025  챠밍헤어클럽미용실  이미용업  이일례  원성동  원성2동   
4   5  천안-2012-029      선경세탁소   세탁업  홍성찬  봉명동   봉명동   

                       소재지주소     데이터기준일자  
0      천안시 동남구 터미널3길 21(신부동)  2026-03-26  
1  천안시 동남구 공설시장2길 3, 3층(대흥동)  2026-03-26  
2       천안시 동남구 대흥로 271(대흥동)  2026-03-26  
3       천안시 동남구 고재4길 55(원성동)  2026-03-26  
4       천안시 동남구 양지4길 15(봉명동)  2026-03-26  

데이터 타입:
연번         int64
지정번호         str
업소명          str
업종           str
대표자          str
읍면동          str
행정동          str
소재지주소        str
데이터기준일자      str
dtype: object



✓ 총 1개의 테이블 로드 완료
Supabase PostgreSQL 연결 중...

✓ PostgreSQL 연결 

쿼리 결과가 비어 있어, 현재 제공된 정보만으로는 질문에 대한 구체적인 답변을 드릴 수 없습니다.

참고로 실행한 SQL은 테이블의 **컬럼 구조만 확인**하고 실제 데이터는 가져오지 않았습니다(`LIMIT 0`).  
즉, `"충청남도 천안시_착한가격업소_20260326"` 테이블의 **데이터 행이 조회되지 않은 상태**입니다.

원하시면 다음 중 하나를 해보실 수 있습니다:
1. 실제 데이터가 나오도록 `LIMIT 10` 같은 쿼리로 다시 조회
2. 질문 내용을 구체적으로 알려주기
3. 필요한 컬럼만 선택해서 다시 조회하기

예:
```sql
SELECT *
FROM "충청남도 천안시_착한가격업소_20260326"
LIMIT 10;
```

데이터가 포함된 결과를 주시면 바로 답변드리겠습니다.


질문: 천안시 착한가격 업소는 총 몇개인가요?

[1] SQL 생성 중...
    SELECT COUNT(*) AS "총업소수"
FROM "충청남도 천안시_착한가격업소_20260326";

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


천안시 착한가격 업소는 **총 124개**입니다.


질문: 천안시 착한가격 업소 중 한식 업종은 몇개인가요?

[1] SQL 생성 중...
    SELECT COUNT(*) AS 한식업종수
FROM "충청남도 천안시_착한가격업소_20260326"
WHERE "업종" = '한식';

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


천안시 착한가격 업소 중 **한식 업종은 60개**입니다.


질문: 천안시 착한가격 업소 중 중식 업종의 상호명과 주소를 알려주세요.

[1] SQL 생성 중...
    SELECT "업소명", "소재지주소"
FROM "충청남도 천안시_착한가격업소_20260326"
WHERE "업종" = '중식';

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


천안시 착한가격 업소 중 **중식 업종**의 상호명과 주소는 다음과 같습니다.

1. **청룡각** — 천안시 서북구 충무로 143-8(쌍용동)  
2. **북경중화요리** — 천안시 동남구 신부1길 6(신부동)  
3. **명윤** — 천안시 서북구 직산읍 4산단로 241, 1동  
4. **강짬뽕** — 천안시 동남구 신부12길 12, 1층(신부동)  
5. **성환반점** — 천안시 서북구 성환읍 성진로 27  
6. **홍콩앤홍교** — 천안시 서북구 한들1로 145(백석동)  

원하시면 제가 이를 **표 형태**로도 정리해드릴게요.